# UAV-RIS hierarchical RL: paper experiments
Runtime -> Change runtime type -> GPU. Run the cells in order. Every stage writes JSON/PDF under `results/`; the last cell zips them. See `PAPER_PLAN.md` for what each result means and the decision rules.

In [1]:
!git clone https://github.com/jk-afridi/UAV-RIS-IoT-Hierarchical-RL.git
%cd UAV-RIS-IoT-Hierarchical-RL
!git checkout feat/paper-v2
!pip -q install pytest
import torch; print(torch.__version__, torch.cuda.is_available())

Cloning into 'UAV-RIS-IoT-Hierarchical-RL'...
remote: Enumerating objects: 359, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 359 (delta 24), reused 76 (delta 15), pack-reused 267 (from 1)
Receiving objects: 100% (359/359), 71.27 MiB | 17.16 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/UAV-RIS-IoT-Hierarchical-RL
branch 'feat/paper-v2' set up to track 'origin/feat/paper-v2'.
Switched to a new branch 'feat/paper-v2'
2.11.0+cu130 True


## 0. Sanity: tests (must all pass before trusting anything)

In [2]:
!bash scripts/run_colab.sh tests

...........                                                              [100%]
11 passed in 9.31s


## 1. Baseline benchmark in the fixed environment (CPU, a few minutes)

In [3]:
!bash scripts/run_colab.sh baselines

method                                    avg AoI         peak AoI         decode %  (mean +- 95% CI over 5 seeds)
greedy_aoi_travel+ao_aware          10.03 +- 3.30    49.33 +-18.75    71.28 +-15.52
greedy_aoi_travel+ao_unaware        14.96 +- 4.31    68.60 +-15.63    52.94 +-10.76
greedy_aoi_travel+best_of_5         21.67 +- 4.65    90.93 +-17.95    39.75 +- 6.26
greedy_aoi_travel+random1           39.54 +-11.95   151.53 +-36.45    21.04 +- 3.82
greedy_aoi+ao_aware                  9.62 +- 2.54    58.80 +-25.06    67.38 +-14.58
round_robin+ao_aware                13.95 +- 6.30   159.87 +-67.23    72.27 +-13.61
random+random1                      80.47 +-11.26   201.00 +- 0.00     5.26 +- 2.72
saved results/eval_v2/baselines.json  (312s)


## 2. Mismatch study WITHOUT learning (does AO degrade when the hardware deviates?)
If `ao_nominal` is within noise of `ao_calibrated` at every spread, mismatch is not a problem worth solving -> read PAPER_PLAN.md, branch B.

In [4]:
!bash scripts/run_colab.sh mismatch_base

  spread 0.0 seed 0 done (18s)
  spread 0.0 seed 1 done (36s)
  spread 0.0 seed 2 done (54s)
  spread 0.0 seed 3 done (72s)
  spread 0.0 seed 4 done (90s)
  spread 0.5 seed 0 done (108s)
  spread 0.5 seed 1 done (126s)
  spread 0.5 seed 2 done (144s)
  spread 0.5 seed 3 done (162s)
  spread 0.5 seed 4 done (180s)
  spread 1.0 seed 0 done (198s)
  spread 1.0 seed 1 done (216s)
  spread 1.0 seed 2 done (235s)
  spread 1.0 seed 3 done (253s)
  spread 1.0 seed 4 done (270s)
  spread 2.0 seed 0 done (288s)
  spread 2.0 seed 1 done (305s)
  spread 2.0 seed 2 done (323s)
  spread 2.0 seed 3 done (340s)
  spread 2.0 seed 4 done (358s)

=== hardware spread 0.0  (mean +- 95% CI over 5 seeds)
method          probes         decode %         min rate
ao_nominal           0     75.5 +-  7.2    0.760 +-0.272
ao_ideal             0     50.7 +-  7.9    0.276 +-0.064
ao_calibrated        0     74.5 +-  6.1    0.750 +-0.244
cem6                 6     69.8 +-  6.4    0.605 +-0.244
cem15               15  

## 3. Does the residual policy learn anything? (8k slots, one seed)
In the log, `gain` = policy reward - AO-prior reward on fresh slots. It must become clearly positive.

In [5]:
!bash scripts/run_colab.sh residual_try

slots    1000 | policy 1.832 | AO-prior 1.835 | gain -0.004 | 16s
slots    2000 | policy 1.790 | AO-prior 1.781 | gain +0.009 | 36s
slots    3000 | policy 1.910 | AO-prior 1.910 | gain +0.000 | 56s
slots    4000 | policy 1.876 | AO-prior 1.874 | gain +0.002 | 77s
slots    5000 | policy 1.696 | AO-prior 1.710 | gain -0.014 | 98s
slots    6000 | policy 1.599 | AO-prior 1.629 | gain -0.029 | 118s
slots    7000 | policy 1.710 | AO-prior 1.764 | gain -0.054 | 139s
slots    8000 | policy 1.747 | AO-prior 1.791 | gain -0.044 | 159s
saved results/residual/seed0_spread1.0


## 4. Full mismatch study with learning (long: run in a session you can leave open)

In [6]:
!bash scripts/run_colab.sh mismatch_full

  spread 0.0 seed 0 done (602s)
  spread 0.0 seed 1 done (1206s)
  spread 0.0 seed 2 done (1805s)
  spread 0.0 seed 3 done (2412s)
  spread 0.0 seed 4 done (3030s)
  spread 0.5 seed 0 done (3651s)
  spread 0.5 seed 1 done (4260s)
  spread 0.5 seed 2 done (4865s)
  spread 0.5 seed 3 done (5475s)
  spread 0.5 seed 4 done (6085s)
  spread 1.0 seed 0 done (6105s)
  spread 1.0 seed 1 done (6718s)
  spread 1.0 seed 2 done (7330s)
  spread 1.0 seed 3 done (7935s)
  spread 1.0 seed 4 done (8539s)
  spread 2.0 seed 0 done (9154s)
  spread 2.0 seed 1 done (9773s)
  spread 2.0 seed 2 done (10395s)
  spread 2.0 seed 3 done (11016s)
  spread 2.0 seed 4 done (11637s)

=== hardware spread 0.0  (mean +- 95% CI over 5 seeds)
method          probes         decode %         min rate
ao_nominal           0     75.0 +-  7.5    0.775 +-0.286
ao_ideal             0     51.8 +-  6.8    0.288 +-0.061
ao_calibrated        0     75.5 +-  6.9    0.766 +-0.255
cem6                 6     68.0 +-  7.2    0.588 +-0.2

## 5. Macro PPO (cluster choice + trajectory) and the episode-level AoI study

In [7]:
!bash scripts/run_colab.sh macro

ep   100 | held-out AoI  PPO  85.79  greedy  10.87  (PPO-greedy +74.91) | train AoI  25.74 | 255s
ep   200 | held-out AoI  PPO  84.70  greedy  11.10  (PPO-greedy +73.60) | train AoI  20.82 | 499s
ep   300 | held-out AoI  PPO  70.96  greedy  11.30  (PPO-greedy +59.66) | train AoI  20.38 | 746s
ep   400 | held-out AoI  PPO  41.55  greedy  10.98  (PPO-greedy +30.57) | train AoI  19.85 | 996s
ep   500 | held-out AoI  PPO  38.27  greedy  10.67  (PPO-greedy +27.60) | train AoI  19.53 | 1251s
ep   600 | held-out AoI  PPO  30.62  greedy  10.54  (PPO-greedy +20.08) | train AoI  22.44 | 1503s
ep   700 | held-out AoI  PPO  15.58  greedy  10.91  (PPO-greedy +4.67) | train AoI  25.54 | 1753s
ep   800 | held-out AoI  PPO  18.49  greedy  10.91  (PPO-greedy +7.58) | train AoI  19.88 | 2004s
ep   900 | held-out AoI  PPO  16.53  greedy  11.18  (PPO-greedy +5.34) | train AoI  18.66 | 2254s
ep  1000 | held-out AoI  PPO  21.67  greedy  11.04  (PPO-greedy +10.63) | train AoI  16.32 | 2500s
ep  1100 | held-o

In [8]:
!bash scripts/run_colab.sh aoi

seed 0 done (84s)
seed 1 done (167s)
seed 2 done (251s)
seed 3 done (335s)
seed 4 done (419s)
scheduler+lowlevel                            avg AoI         peak AoI         decode %
greedy_aoi_travel+ao_nominal            11.63 +- 3.41    55.60 +-14.33    64.65 +-14.15
greedy_aoi+ao_nominal                   11.95 +- 3.26    66.27 +-17.86    56.96 +-11.97
round_robin+ao_nominal                  16.13 +- 6.84   161.33 +-60.97    65.22 +-13.06
random+ao_nominal                       62.37 +-15.84   200.80 +- 0.56    15.12 +- 5.68
ppo+ao_nominal                          27.63 +-25.05   124.33 +-81.63    38.71 +-24.21
greedy_aoi_travel+ao_calibrated         10.75 +- 3.55    52.13 +-16.75    67.90 +-15.37
greedy_aoi+ao_calibrated                10.67 +- 2.83    62.47 +-12.70    61.61 +-13.63
round_robin+ao_calibrated               15.44 +- 7.07   157.07 +-66.04    68.75 +-13.49
random+ao_calibrated                    61.30 +-15.39   199.07 +- 5.37    16.39 +- 6.10
ppo+ao_calibrated         

## 6. Figures + LaTeX tables, then download everything

In [9]:
!bash scripts/run_colab.sh report
!zip -rq results.zip results
from google.colab import files; files.download('results.zip')

report written to results/report


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>